Split podcast episodes into segments and transcribe each one. Uses `audio_splitter` to break long files into manageable chunks, then Whisper for local transcription — no API key needed.

## Problem

You have long podcast episodes that need to be segmented for transcription, indexing, or clip extraction. Long audio files are too large to process at once — Whisper and other transcription models work best on shorter segments. You need to split the audio into chunks before transcription.

## Solution

**What's in this recipe:**

- Split podcast episodes into segments with `audio_splitter`
- Transcribe each segment with Whisper (runs locally, no API key)
- View transcriptions with timestamps

### Setup

In [1]:
%pip install -qU pixeltable openai-whisper

Note: you may need to restart the kernel to use updated packages.


In [2]:
import pixeltable as pxt
from pixeltable.functions import whisper
from pixeltable.functions.audio import audio_splitter

### Load a podcast episode

In [3]:
pxt.drop_dir('podcast_demo', force=True)
pxt.create_dir('podcast_demo')

Connected to Pixeltable database at: postgresql+psycopg://postgres:@/pixeltable?host=/Users/alison-pxt/.pixeltable/pgdata
Created directory 'podcast_demo'.


In [4]:
episodes = pxt.create_table(
    'podcast_demo/episodes',
    {'title': pxt.String, 'audio': pxt.Audio}
)

Created table 'episodes'.


In [5]:
# Insert a sample podcast excerpt (video files work too — audio is extracted automatically)
episodes.insert([{
    'title': 'Lex Fridman Podcast Excerpt',
    'audio': 'https://raw.githubusercontent.com/pixeltable/pixeltable/main/docs/resources/audio-transcription-demo/Lex-Fridman-Podcast-430-Excerpt-0.mp4'
}])

Inserted 1 row with 0 errors in 0.36 s (2.81 rows/s)


1 row inserted.

### Split into segments

Create a view that splits audio into 30-second segments with overlap. The overlap helps avoid losing words at segment boundaries.

In [6]:
segments = pxt.create_view(
    'podcast_demo/segments',
    episodes,
    iterator=audio_splitter(
        episodes.audio,
        duration=30.0,
        overlap=2.0,
        min_segment_duration=5.0,
    ),
)

In [7]:
# View the segments
segments.select(
    segments.segment_start,
    segments.segment_end
).collect()

segment_start,segment_end
0.,30.
28.003,58.003


### Transcribe each segment

Add a computed column for transcription. Whisper runs locally on each segment.

In [8]:
segments.add_computed_column(
    transcription=whisper.transcribe(
        audio=segments.audio_segment,
        model='base.en',
    )
)

Added 2 column values with 0 errors in 4.01 s (0.50 rows/s)


2 rows updated.

In [9]:
segments.add_computed_column(text=segments.transcription.text)

Added 2 column values with 0 errors in 0.03 s (67.11 rows/s)


2 rows updated.

In [10]:
# View transcriptions with timestamps
segments.select(
    segments.segment_start,
    segments.segment_end,
    segments.text
).collect()

segment_start,segment_end,text
0.,30.,"of experiencing self versus remembering self. I was hoping you can give a simple answer of how we should live life. Based on the fact that our memories could be a source of happiness or could be the primary source of happiness, that an event when experienced bears its fruits the most when it's remembered over and over and over and over."
28.003,58.003,"over and over and over and over and maybe there is some wisdom in the fact that we can control to some degree how we remember how we evolve our memory of it such that it can maximize the long-term happiness of that repeated experience. Okay, well first I'll say I wish I could take you on the road with me. That was such a great description. Can I be your opening ax? Oh my God, no, I'm going to open for you dude. Otherwise it's like, you know, everybody leaves."


### Get audio metadata

You can also inspect metadata about the original audio file — codec, duration, sample rate, channels.

In [11]:
from pixeltable.functions.audio import get_metadata

episodes.add_computed_column(metadata=get_metadata(episodes.audio))
episodes.select(episodes.title, episodes.metadata).collect()

title,metadata
Lex Fridman Podcast Excerpt,"{""size"": 6465748, ""streams"": [{""type"": ""video"", ""width"": 1920, ""frames"": 1800, ""height"": 1080, ""duration"": 921600, ""metadata"": {""encoder"": ""Lavc61.3.100 libx264"", ""language"": ""und"", ""vendor_id"": ""[0][0][0][0]"", ""handler_name"": ""ISO Media file produced by Google Inc.""}, ""base_rate"": 30., ""time_base"": 6.51e-05, ""average_rate"": 30., ""guessed_rate"": 30., ""codec_context"": {""name"": ""h264"", ""pix_fmt"": ""yuv420p"", ""profile"": ""High"", ""codec_tag"": ""avc1""}, ""duration_seconds"": 60.}, {""type"": ""audio"", ""frames"": 2585, ""duration"": 2646000, ""metadata"": {""language"": ""eng"", ""vendor_id"": ""[0][0][0][0]"", ""handler_name"": ""ISO Media file produced by Google Inc.""}, ""time_base"": 2.268e-05, ""codec_context"": {""name"": ""aac"", ""profile"": ""LC"", ""channels"": 2, ""codec_tag"": ""mp4a""}, ""duration_seconds"": 60.}], ""bit_rate"": 862099, ""metadata"": {""encoder"": ""Lavf61.1.100"", ""major_brand"": ""isom"", ""minor_version"": ""512"", ""compatible_brands"": ""isomiso2avc1mp41""}, ""bit_exact"": false}"


## Explanation

**Pipeline architecture:**

```
Audio → audio_splitter(duration=30) → segments → Whisper → transcripts
```

Each step is a computed column or view. When you insert a new episode, all steps run automatically.

**`audio_splitter` parameters:**

| Parameter | Description | Default |
|-----------|-------------|---------|
| `duration` | Length of each segment in seconds | (required) |
| `overlap` | Overlap between consecutive segments in seconds | 0.0 |
| `min_segment_duration` | Drop the last segment if shorter than this | 0.0 |

**Choosing a segment duration:**

- **30 seconds** is a good starting point for most podcasts
- Shorter segments (10-15s) give finer-grained timestamps but may split mid-sentence
- Longer segments (60s+) preserve more context but use more memory during transcription
- The `overlap` parameter helps avoid losing words at segment boundaries

**Whisper model options:**

| Model | Speed | Quality | Best for |
|-------|-------|---------|----------|
| `tiny.en` | Fastest | Basic | Quick tests |
| `base.en` | Fast | Good | General use |
| `small.en` | Medium | Better | Higher accuracy |
| `medium.en` | Slow | Great | Professional quality |
| `large` | Slowest | Best | Maximum accuracy |

Models ending in `.en` are English-only and faster. Remove `.en` for multilingual support.

## See also

- [Transcribe audio](https://docs.pixeltable.com/howto/cookbooks/audio/audio-transcribe) — Basic audio transcription
- [Summarize podcasts](https://docs.pixeltable.com/howto/cookbooks/audio/audio-summarize-podcast) — Transcription + LLM summarization
- [Extract audio from video](https://docs.pixeltable.com/howto/cookbooks/audio/audio-extract-from-video) — Working with video files